# HousePrice AI — Phase 7: Model Training & Convergence Analysis

## 1. Objectives
In this notebook, we train our custom Multiple Linear Regression model on the housing dataset:
1. Load preprocessed feature splits.
2. Standardize target values $y$ to enhance gradient descent numerical stability.
3. Train with Batch Gradient Descent while tracking both training and validation losses.
4. Diagnose convergence and plot training curves.
5. Save the trained model and preprocessing scalers to `models/linear_regression.json`.

---

In [21]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

sys.path.append(os.path.abspath('..'))

from src.preprocessing import StandardScaler
from src.linear_regression import LinearRegressionScratch
from src.metrics import evaluate_all, print_metrics
from src.visualization import plot_loss_curve, plot_train_val_loss
from src.model_io import save_model

## 2. Load Processed Data Splits

In [22]:
splits_file = '../data/processed/splits_baseline.npz'
if os.path.exists(splits_file):
    data = np.load(splits_file, allow_pickle=True)
    X_train = data['X_train']
    X_val = data['X_val']
    X_test = data['X_test']
    y_train = data['y_train']
    y_val = data['y_val']
    y_test = data['y_test']
    feature_names = list(data['features'])
else:
    print('Processed splits not found; generating on the fly...')
    from src.data_loader import load_dataset
    from src.preprocessing import handle_missing_values, remove_missing_target
    from src.data_split import train_val_test_split
    df = load_dataset('../data/raw/train.csv' if os.path.exists('../data/raw/train.csv') else '../data/House Price Prediction Dataset.csv')
    target_col = 'SalePrice' if 'SalePrice' in df.columns else 'Price'
    df = remove_missing_target(df, target_col)
    feature_names = [c for c in ['GrLivArea', 'OverallQual', 'YearBuilt', 'BedroomAbvGr', 'FullBath', 'GarageCars'] if c in df.columns]
    X = df[feature_names].values.astype(np.float64)
    y = df[target_col].values.astype(np.float64)
    X_train, X_val, X_test, y_train, y_val, y_test = train_val_test_split(X, y, 0.7, 0.15, 0.15, 42)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)

print(f'Train shape: {X_train.shape}, Val shape: {X_val.shape}, Test shape: {X_test.shape}')
print(f'Features: {feature_names}')

Train shape: (1021, 6), Val shape: (219, 6), Test shape: (220, 6)
Features: [np.str_('GrLivArea'), np.str_('OverallQual'), np.str_('YearBuilt'), np.str_('BedroomAbvGr'), np.str_('FullBath'), np.str_('GarageCars')]


## 3. Target Variable Standardization

House prices are on the scale of \$100,000–\$700,000. Large target values cause gradients $2X^T(\hat{y} - y)$ to be immense ($10^7$ or higher), forcing an impractically small learning rate.
Standardizing $y$ into $z_y \sim \mathcal{N}(0, 1)$ stabilizes optimization.

In [23]:
scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).ravel()
y_val_scaled = scaler_y.transform(y_val.reshape(-1, 1)).ravel()

print(f'Target mean: {scaler_y.mean[0]:.2f}')
print(f'Target std:  {scaler_y.std[0]:.2f}')

Target mean: 180975.72
Target std:  82217.56


## 4. Model Training

In [24]:
model = LinearRegressionScratch(learning_rate=0.01, n_epochs=1000, l2_lambda=0.0)
model.fit(X_train, y_train_scaled, X_val=X_val, y_val=y_val_scaled, verbose=True, print_every=100)

TRAINING LINEAR REGRESSION
  Samples       : 1021
  Features      : 6
  Learning rate : 0.01
  Epochs        : 1000
  L2 lambda     : 0.0
------------------------------------------------------------
  Epoch     0/1000  |  Train Loss: 1.000000  |  Val Loss: 0.796370
  Epoch   100/1000  |  Train Loss: 0.269445  |  Val Loss: 0.186751
  Epoch   200/1000  |  Train Loss: 0.259591  |  Val Loss: 0.173505
  Epoch   300/1000  |  Train Loss: 0.257643  |  Val Loss: 0.169985
  Epoch   400/1000  |  Train Loss: 0.257160  |  Val Loss: 0.168689
  Epoch   500/1000  |  Train Loss: 0.257013  |  Val Loss: 0.168088
  Epoch   600/1000  |  Train Loss: 0.256962  |  Val Loss: 0.167766
  Epoch   700/1000  |  Train Loss: 0.256942  |  Val Loss: 0.167578
  Epoch   800/1000  |  Train Loss: 0.256934  |  Val Loss: 0.167463
  Epoch   900/1000  |  Train Loss: 0.256931  |  Val Loss: 0.167389
  Epoch   999/1000  |  Train Loss: 0.256930  |  Val Loss: 0.167343
------------------------------------------------------------
[IN

## 5. Visualizing Loss Curves

In [25]:
plot_train_val_loss(model.training_loss_history, model.validation_loss_history,
                    save_path='../outputs/training/train_validation_loss.png')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(model.training_loss_history, label='Training Loss', color='steelblue', lw=1.5)
ax.plot(model.validation_loss_history, label='Validation Loss', color='coral', lw=1.5)
ax.set_title('Training vs Validation Loss (Standardized Scale)', fontsize=14, fontweight='bold')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('MSE Loss', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.show()

[SAVED] ../outputs/training/train_validation_loss.png


C:\Users\jayp1\AppData\Local\Temp\ipykernel_30780\756108302.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Model Coefficients & Feature Importance Analysis

In [26]:
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Weight': model.weights,
    'AbsWeight': np.abs(model.weights)
}).sort_values('AbsWeight', ascending=False)

print('Feature Weights (Standardized Coefficients):')
display(importance_df)
print(f'Bias (Intercept): {model.bias:.6f}')

Feature Weights (Standardized Coefficients):


,Feature,Weight,AbsWeight
0,GrLivArea,0.422518,0.422518
1,OverallQual,0.392063,0.392063
5,GarageCars,0.139595,0.139595
2,YearBuilt,0.134867,0.134867
3,BedroomAbvGr,-0.083076,0.083076
4,FullBath,-0.025846,0.025846


Bias (Intercept): -0.000000


## 7. Model Persistence
We save the trained parameters and scalers to `models/linear_regression.json`.

In [27]:
os.makedirs('../models', exist_ok=True)
model_file = '../models/linear_regression.json'
save_model(
    model=model,
    scaler_X=StandardScaler().fit(X_train), # feature scaler
    scaler_y=scaler_y,
    feature_names=feature_names,
    filepath=model_file,
    config={'learning_rate': 0.01, 'n_epochs': 1000}
)
print(f'[SUCCESS] Model successfully saved to {model_file}')

[SUCCESS] Model saved to: ../models/linear_regression.json
   Weights : 6 features
   Bias    : -0.000000
[SUCCESS] Model successfully saved to ../models/linear_regression.json
